# La Programmation Asynchrone en Python avec `asyncio`

Une fonction asynchrone est une fonction  qui peut s’interrompre volontairement à certains endroits pour laisser d’autres tâches s’exécuter, puis reprendre plus tard exactement où elle s’était arrêtée.

#### Quelle différence avec les `générateurs` et le `threading` ?


Alors pourquoi s'intéresser à une technique de plus ? Que nous apporterait-elle ?

- Les `generateurs` permettent certes de créer des coroutines avec la méchanique `yield`, mais pas de les orchestrer automatiquement pour optimiser les temps d'attente d'un programme...
- Le `threading` est efficace pour les tâches paralleles avec des temps d'attente, mais trop de threads peuvent rendre le tout difficile à orchestrer...
- `asyncio` est la solution permettant d’orchestrer efficacement un volume élevé de coroutines au sein d’un unique thread via un modèle événementiel.


C’est pourquoi `asyncio` est aujourd’hui utile pour :

- serveurs web haute performance (FastAPI, aiohttp)
- clients HTTP massifs
- synchronisation réseau
- pipelines I/O
- programmation événementielle

https://docs.python.org/3/library/asyncio.html

# Le coeur de `asyncio` : **L'Event Loop**

Au coeur de `asyncio` on retrouve un composant nommé `Event Loop` dont le role est d'orchestrer les différentes coroutines qui pourraient exister dans notre code.

Pour faire simple, son role est de s'assurer qu'il y ait toujours du travail a réaliser dans notre programme python, sans temps d'attente.
Il va donc permettre de gerer plusieurs coroutines, un peu comme si nous avions plusieurs `threads` sauf qu'ici, il n'y en a qu'un seul !

Pour s'en servir, il faut donc créer des coroutines avec la syntaxe suivante : 

### Syntaxe
```python
async def ma_coroutine():
    ....
```


Ensuite, une fois notre fonction créée, on peut la lancer de la maniere suivante : 

```python
await ma_coroutine() # si nous sommes dans un notebook .ipynb

# ou bien...

asyncio.run(ma_coroutine()) # si nous sommes dans un fichier .py
```

Exemple 

In [1]:
import asyncio

async def hello():
    print("Hello asyncio !")

In [2]:
hello()

<coroutine object hello at 0x7b8ed3d2b640>

In [3]:
await hello()
# asyncio.run(hello())

Hello asyncio !


### Mettre en pause l’exécution d’une coroutine

- **`async def`** : Définit une fonction asynchrone (coroutine), qui peut contenir des suspensions.
- **`await`** : Met en pause l’exécution d’une coroutine jusqu’à ce qu’une tâche asynchrone soit terminée.

### Syntaxe
```python
async def ma_coroutine():
    await une_tache()
```

#### exemple : 

In [4]:
async def fetch_data():
    print("chargement de données en cours...")
    await asyncio.sleep(2) # simuler une opération I/O
    print("Données recues !")
    return {"data": [1, 2, 3, 4, 5]}

In [ ]:
async def main():
    print("Démarrage de la coroutine principale")

    task = fetch_data() # on créer une tache de coroutine, mais celle-ci ne s'execute pas encore
    results = await task # ici on execute la coroutine, et on obtient ces résultats en sortie
    
    print(f"Les résultats : {results}")
    print("Programme terminé")

In [6]:
await main()

Démarrage de la coroutine principale
chargement de données en cours...
Données recues !
Les résultats : {'data': [1, 2, 3, 4, 5]}
Programme terminé


Si vous avez bien compris... pourquoi doit-on utiliser `await main()` dans le jupyter notebook ?

#### Un exemple avec cette fois-ci plusieurs coroutines `fetch_data`

In [8]:
async def main():
    print("Démarrage de la coroutine principale")
    task1 = fetch_data()
    task2 = fetch_data()

    results1 = await task1
    results2 = await task2

    print(f"Les résultats : {results1}")
    print(f"Les résultats : {results2}")
    print("Programme terminé")

In [9]:
await main()

Démarrage de la coroutine principale
chargement de données en cours...
Données recues !
chargement de données en cours...
Données recues !
Les résultats : {'data': [1, 2, 3, 4, 5]}
Les résultats : {'data': [1, 2, 3, 4, 5]}
Programme terminé


Ici, les coroutines **ne sont pas concurrentes !** (ca n'est pas comme si nous avions plusieurs threads..) et nous n'avons donc aucun gain de temps !

# Comment executer des coroutines en "parallele" ?

Le module **`asyncio`** fournit une boucle d’événements (*event loop*) pour exécuter des coroutines de manière concurrente. Il s'agit d'une sorte d'orchestrateur, qui s'assure que rien n'est jamais bloqué (en attente) dans votre code. Si une fonctionne "dort" il passe la main a une autre.

Pour ce faie, nous allons utiliser ce qu'on appelle des `tasks`
- Soit avec la fonction `asyncio.create_task()`
- Soit avec **`asyncio.gather()`** : qui est bien plus simple et direct
- Soit avec `TastGroup`, qui permet de gerer les exceptions d'une tache sans bloquer les autres

#### Exemple

In [10]:
import asyncio


async def fetch_data(nom: str, delai: float):
    print(f"{nom} Lancée ! chargement de données en cours ...")
    await asyncio.sleep(delai) # simuler une opération I/O
    print("Données recues !")
    return {"data": [1, 2, 3, 4, 5]}


async def main():
    """Exécute plusieurs tâches concurremment."""
    task1 = asyncio.create_task(fetch_data("Tâche 1", 2))
    task2 = asyncio.create_task(fetch_data("Tâche 2", 0.5))
    task3 = asyncio.create_task(fetch_data("Tâche 3", 1))

    result1 = await task1
    result2 = await task2
    result3 = await task3

    print("Résultats 1:", result1)
    print("Résultats 2:", result2)
    print("Résultats 3:", result3)



In [11]:
await main()

Tâche 1 Lancée ! chargement de données en cours ...
Tâche 2 Lancée ! chargement de données en cours ...
Tâche 3 Lancée ! chargement de données en cours ...
Données recues !
Données recues !
Données recues !
Résultats 1: {'data': [1, 2, 3, 4, 5]}
Résultats 2: {'data': [1, 2, 3, 4, 5]}
Résultats 3: {'data': [1, 2, 3, 4, 5]}


#### Meme Exemple, cette fois-ci avec `gather` : 

- **`asyncio.gather`** : Lance les trois tâches en parallèle ; elles se terminent selon leurs délais.
- **Concurrence** : Les tâches s’exécutent simultanément dans une seule boucle d’événements, pas en parallèle comme avec des threads.
- **Non bloquant** : Le programme reste réactif pendant les attentes.

In [12]:
import asyncio


async def fetch_data(nom: str, delai: float):
    print(f"{nom} Lancée ! chargement de données en cours ...")
    await asyncio.sleep(delai) # simuler une opération I/O
    print("Données recues !")
    return {"data": [1, 2, 3, 4, 5]}


async def main():
    """Exécute plusieurs tâches concurremment."""
    resultats = await asyncio.gather(
        fetch_data("Tâche 1", 2),
        fetch_data("Tâche 2", 1),
        fetch_data("Tâche 3", 1.5)
    )
    print("Résultats :", resultats)


In [13]:
await main()

Tâche 1 Lancée ! chargement de données en cours ...
Tâche 2 Lancée ! chargement de données en cours ...
Tâche 3 Lancée ! chargement de données en cours ...
Données recues !
Données recues !
Données recues !
Résultats : [{'data': [1, 2, 3, 4, 5]}, {'data': [1, 2, 3, 4, 5]}, {'data': [1, 2, 3, 4, 5]}]


#### Exemple avec `TaskGroup`

In [14]:
import asyncio


async def fetch_data(nom: str, delai: float):
    print(f"{nom} Lancée ! chargement de données en cours ...")
    await asyncio.sleep(delai) # simuler une opération I/O
    print("Données recues !")
    return {"data": [1, 2, 3, 4, 5]}


async def main():
    """Exécute plusieurs tâches concurremment."""
    async with asyncio.TaskGroup() as tg:
        task1 = tg.create_task(fetch_data("Tache 1", 1))
        task2 = tg.create_task(fetch_data("Tache 2", 1.4))
        task3 = tg.create_task(fetch_data("Tache 3", 0.5))

    results1 = task1.result()
    results2 = task2.result()
    results3 = task3.result()

    print(results1, results2, results3)

In [15]:
await main()

Tache 1 Lancée ! chargement de données en cours ...
Tache 2 Lancée ! chargement de données en cours ...
Tache 3 Lancée ! chargement de données en cours ...
Données recues !
Données recues !
Données recues !
{'data': [1, 2, 3, 4, 5]} {'data': [1, 2, 3, 4, 5]} {'data': [1, 2, 3, 4, 5]}


# Utilisation de Lock

Comme pour les `threads` et les `processus`, on peut **vérouiller** certaines opérations afin d'éviter les problemes de coroutines. La syntaxe et la logique est **exactement** la meme !

In [20]:
class MontantInvalidError(Exception):
    pass

In [21]:
class CompteBancaire:
    def __init__(self, solde_initial: int):
        self.solde = solde_initial
        self._lock = asyncio.Lock()

    async def retirer(self, montant: int) -> int:
        print(f"Demande de retrait : {montant}")

        async with self._lock:
            if montant > self.solde:
                raise MontantInvalidError(f"Solde insuffisant : {self.solde}, demandé {montant}")

            await asyncio.sleep(2)  # simulation du temps de traitement
            self.solde -= montant
            print(f"Retrait réussi : {montant}, nouveau solde : {self.solde}")
            return montant

In [22]:
async def main():
    compte = CompteBancaire(100_000)

    async with asyncio.TaskGroup() as tg:
        t1 = tg.create_task(compte.retirer(20_000))
        t2 = tg.create_task(compte.retirer(40_000))
        t3 = tg.create_task(compte.retirer(50_000))

    print("Résultats :", t1.result(), t2.result(), t3.result())

In [23]:
await main()

Demande de retrait : 20000
Demande de retrait : 40000
Demande de retrait : 50000
Retrait réussi : 20000, nouveau solde : 80000
Retrait réussi : 40000, nouveau solde : 40000


  + Exception Group Traceback (most recent call last):
  |   File "/home/guillaume/.pyenv/versions/3.12.9/lib/python3.12/site-packages/IPython/core/interactiveshell.py", line 3665, in run_code
  |     await eval(code_obj, self.user_global_ns, self.user_ns)
  |   File "/tmp/ipykernel_206569/1359770034.py", line 1, in <module>
  |     await main()
  |   File "/tmp/ipykernel_206569/646348658.py", line 4, in main
  |     async with asyncio.TaskGroup() as tg:
  |                ^^^^^^^^^^^^^^^^^^^
  |   File "/home/guillaume/.pyenv/versions/3.12.9/lib/python3.12/asyncio/taskgroups.py", line 71, in __aexit__
  |     return await self._aexit(et, exc)
  |            ^^^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "/home/guillaume/.pyenv/versions/3.12.9/lib/python3.12/asyncio/taskgroups.py", line 164, in _aexit
  |     raise BaseExceptionGroup(
  | ExceptionGroup: unhandled errors in a TaskGroup (1 sub-exception)
  +-+---------------- 1 ----------------
    | Traceback (most recent call last):
    |   Fi

In [24]:
try:
    await main()
except MontantInvalidError:
    print("Erreur")

Demande de retrait : 20000
Demande de retrait : 40000
Demande de retrait : 50000
Retrait réussi : 20000, nouveau solde : 80000
Retrait réussi : 40000, nouveau solde : 40000


  + Exception Group Traceback (most recent call last):
  |   File "/home/guillaume/.pyenv/versions/3.12.9/lib/python3.12/site-packages/IPython/core/interactiveshell.py", line 3665, in run_code
  |     await eval(code_obj, self.user_global_ns, self.user_ns)
  |   File "/tmp/ipykernel_206569/3867855103.py", line 2, in <module>
  |     await main()
  |   File "/tmp/ipykernel_206569/646348658.py", line 4, in main
  |     async with asyncio.TaskGroup() as tg:
  |                ^^^^^^^^^^^^^^^^^^^
  |   File "/home/guillaume/.pyenv/versions/3.12.9/lib/python3.12/asyncio/taskgroups.py", line 71, in __aexit__
  |     return await self._aexit(et, exc)
  |            ^^^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "/home/guillaume/.pyenv/versions/3.12.9/lib/python3.12/asyncio/taskgroups.py", line 164, in _aexit
  |     raise BaseExceptionGroup(
  | ExceptionGroup: unhandled errors in a TaskGroup (1 sub-exception)
  +-+---------------- 1 ----------------
    | Traceback (most recent call last):
    |   Fi

In [25]:

try:
    await main()
except* MontantInvalidError as eg:
    for exc in eg.exceptions: # eg.exceptions = liste des exceptions réelles
        print("Erreur :", exc)


Demande de retrait : 20000
Demande de retrait : 40000
Demande de retrait : 50000
Retrait réussi : 20000, nouveau solde : 80000
Retrait réussi : 40000, nouveau solde : 40000
Erreur : Solde insuffisant : 40000, demandé 50000


# Utilisation des `Semaphores`

Comme nous l'avons déja vu avec les `threads` et `processus` les sémaphores permettent de limiter le nombre de coroutines dans un meme systeme... ce qui peut par exemple etre tres important pour par exemple eviter de saturer un serveur ou un autre service (si trop de coroutines essaient d'y acceder en meme temps)

In [26]:
import asyncio
import random


async def fetch_data(semaphore, nom):
    async with semaphore:
        print(f"{nom} Lancée ! chargement de données en cours ...")
        sleep_time = random.sample(population=[0.5, 1, 2], k=1)[0]
        await asyncio.sleep(sleep_time) # simuler une opération I/O
        print(f"{nom} Terminée ! - Durée {sleep_time} secondes")
        return {"data": [random.randint(1, 100) for _ in range(10)]}


async def main():
    """Exécute plusieurs tâches concurremment."""

    semaphore = asyncio.Semaphore(3) # autorise 3 coroutines a opérer ensemble.

    task_list = [fetch_data(semaphore, f"Tache {i}") for i in range(10)]

    await asyncio.gather(*task_list)


In [27]:
await main()

Tache 0 Lancée ! chargement de données en cours ...
Tache 1 Lancée ! chargement de données en cours ...
Tache 2 Lancée ! chargement de données en cours ...
Tache 1 Terminée ! - Durée 1 secondes
Tache 3 Lancée ! chargement de données en cours ...
Tache 0 Terminée ! - Durée 2 secondes
Tache 2 Terminée ! - Durée 2 secondes
Tache 4 Lancée ! chargement de données en cours ...
Tache 5 Lancée ! chargement de données en cours ...
Tache 3 Terminée ! - Durée 1 secondes
Tache 6 Lancée ! chargement de données en cours ...
Tache 6 Terminée ! - Durée 0.5 secondes
Tache 7 Lancée ! chargement de données en cours ...
Tache 4 Terminée ! - Durée 1 secondes
Tache 8 Lancée ! chargement de données en cours ...
Tache 8 Terminée ! - Durée 0.5 secondes
Tache 9 Lancée ! chargement de données en cours ...
Tache 7 Terminée ! - Durée 1 secondes
Tache 5 Terminée ! - Durée 2 secondes
Tache 9 Terminée ! - Durée 2 secondes


Note : On peut également utiliser les `queues`, `events` etc. comme nous l'avons vu dans les chapitres sur le `threading` et `multiprocessing`

# En Conclusion

`Asyncio` offre une alternative plus puissante et pratique que `threading` ou les generateurs, lorsqu'il s'agit d'effectuer des opérations I/O, car : 
- Il n'y a qu'un seul `thread`
- Avec un Event Loop dont le role est d'orchestrer les routines

Cela vous donne un code plus performant pour les taches purement I/O que le module `threading`.

Si votre travail de Dev implique de créer des APIs, vous serez confronté au fait d'utiliser `Asyncio` ! Revenez donc a ce cours le jour ou vous en avez besoin !

## Exercice

Une usine intelligente utilise plusieurs bras robotisés. Ils doivent récupérer des pièces sur un convoyeur sans se marcher dessus.

- Le convoyeur est une liste partagée contenant 30 pièces.
- 3 bras robotisés sont disponible pour prendre une pièce.
- Un Lock protège le convoyeur.
- On limite le nombre de bras pouvant travailler simultanément pour des raisons de sécurité a 2.

Chaque bras :

- prend une pièce
- la traite (sleep)
- la stocke dans stock_final

Mission : vider entièrement le convoyeur.

In [ ]:
# Votre code ici

## Correction

In [ ]:
import asyncio

convoyeur = list(range(30))
stock_final = []
lock = asyncio.Lock()
semaphore = asyncio.Semaphore(2)

async def prendre_piece(robot_id):
    while True:
        async with semaphore:
            async with lock:
                if not convoyeur:
                    return
                piece = convoyeur.pop(0)
                print(f"Robot {robot_id} prend la pièce {piece}")

            await asyncio.sleep(0.1)

            async with lock:
                stock_final.append(piece)
                print(f"Robot {robot_id} termine la pièce {piece}")

async def main():
    tasks = [asyncio.create_task(prendre_piece(i)) for i in range(3)]
    await asyncio.gather(*tasks)


In [ ]:
await main()